# Lab 08: How to improve retrieval with hybrid search and reranking

**CPU | API key: `OPENAI_API_KEY` (embedding and chat calls) | In class**

## Goal

Build a keyword search and a reranker beside the existing vector search, then decide whether either change earns
its cost. You will see keyword matching rescue one question, fusion lower another, and a chat model reorder a policy
question's candidates correctly while sometimes returning invalid IDs.

Lab 06 gave us a fixed question set and separate retrieval and answer measures. Lab 07 showed why a judge needs
checking too. Here we change the retrieval order and measure the gains and losses on the same questions. A better
rank is useful only if the retrieved text supports the answer we need.

## What you will learn

- How BM25, the classic keyword ranking, scores a chunk from the words it shares with a question.
- How reciprocal rank fusion merges two ranked lists by their ranks instead of their scores.
- How to compare a change with the baseline question by question, and read a sign test on the questions that moved.
- How to rerank candidates with a chat model, and check in code that the reply names only the candidates.
- How to price a retrieval stage in tokens, dollars and seconds.
- How to write a decision from the paired results and the cost.

**In class:** Parts B to D, then Part E up to Checkpoint E (one rerank call). Part A, the ID check after
Checkpoint E, and Parts F to H are self-study.


## Set up the notebook

Run these two cells first, every time; Lab 00 explains each line. **Setup** finds the repository folder (on Colab it
copies the repository and installs the packages first). **Config** reads the settings and finds your key.

In [1]:
# Setup: on Colab, copy the repository and install the packages. Everywhere else they are installed already.
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not Path("/content/rag_for_nlp").exists():
        !git clone -q --depth 1 https://github.com/tatwan/rag_for_nlp.git /content/rag_for_nlp
    %cd /content/rag_for_nlp
    %pip install -q -r requirements.txt

ROOT = next(folder for folder in [Path.cwd(), *Path.cwd().parents] if (folder / "models.env").exists())
sys.path.insert(0, str(ROOT))
print("Repository folder:", ROOT.name)

Repository folder: rag_for_nlp


In [2]:
# Config: settings from models.env (your .env and environment variables win); the key from the environment, .env or Colab Secrets.
from shared.settings import load_settings, load_key, stamp

settings = load_settings()
key_found_in = load_key("OPENAI_API_KEY")

for name, value in stamp(settings).items():
    print(f"{name:18}{value}")
print(f"{'OPENAI_API_KEY':18}{'found in ' + key_found_in if key_found_in else 'not found'}")

chat_model        gpt-4o-mini
reasoning_effort  None
embed_model       text-embedding-3-small
judge_model       gpt-4o-mini
rerank_model      gpt-4o-mini
chat_endpoint     https://api.openai.com/v1
embed_endpoint    https://api.openai.com/v1
run_date          2026-10-27
OPENAI_API_KEY    found in .env


The settings above identify the embedding model, rerank model and endpoints. `JSON_MODE` selects the structured
reply format; `TOKEN_ENCODING` selects the tokenizer used for evidence counts.

Through Part H, the notebook makes 42 rerank calls: one demonstration, 30 development questions, five repeats of
one question, five Atlas questions and one helper comparison. The optional shuffle challenge adds five calls.
It also embeds the 277 corpus records and the search questions; timing and MMR cells make additional searches.
Re-running a cell makes new requests. Charges depend on your configured provider, models and token usage.
Part F calculates the reranking charge from the prices you supply, rather than treating an example price as a quote.

The saved outputs use `gpt-4o-mini` and `text-embedding-3-small`. Checkpoints describe those outputs. Your replies,
rankings and timings can vary; read each changed question before comparing totals.

We reuse the LangChain `Document`, embeddings object and `InMemoryVectorStore` demonstrated directly in Lab 05.
Keyword search uses `bm25s`; fusion and the comparisons are small Python functions built here before they move to
`shared/hybrid.py`. A search service can provide these stages together, but keeping them separate lets us inspect
what each stage changes.


---

## Part A: Score the five policies with BM25 by hand (self-study)

A keyword score should be explainable from the words it counts. We start with five short policies, work through
the score, then use the public `bm25s` API on those same texts. This is a small model of the full-corpus search in Part B.

| ID | Policy text |
|---|---|
| H01 | Atlas passwords can be reset by email. Temporary accounts require a support agent. |
| H02 | Atlas reset links expire after 30 minutes. Request a new link if the old link has expired. |
| H03 | Atlas exports reports as CSV files. Only workspace owners can export all reports. |
| H04 | Boreal reset links expire after 10 minutes. Password reset uses a mobile app. |
| H05 | Atlas invoices are sent monthly. Workspace owners can download invoices from Billing. |

H01 answers how to reset a password; H02 answers when an Atlas link expires. H04 contains a duration too, but for
Boreal. Read the product and the requested fact together before deciding which policy answers a question.

Lab 06 sent a customer's paraphrase of the reset-link question through the vector search (its case G02) and got the
policies in the order H01, H04, H02: the answer, H02 (*Atlas reset links expire after 30 minutes*), came third.

> *When does the password recovery email stop working for Atlas?*

*Predict first:* would a ranking by shared words put H02 higher or lower? Start with the words. `terms()` lowercases a
text and keeps its words; `re.findall(r"\w+", ...)` returns every run of letters and digits:


In [3]:
# policies: the five Atlas and Boreal policies; terms(): a text as a list of lower-case words
import json, math, re
from collections import Counter

def terms(text):
    return re.findall(r"\w+", text.lower())

policies = json.loads((ROOT / "data" / "support-case.json").read_text(encoding="utf-8"))
recovery = "When does the password recovery email stop working for Atlas?"
print("question:", terms(recovery))
print("H02:     ", terms(policies[1]["text"]))

question: ['when', 'does', 'the', 'password', 'recovery', 'email', 'stop', 'working', 'for', 'atlas']
H02:      ['atlas', 'reset', 'links', 'expire', 'after', '30', 'minutes', 'request', 'a', 'new', 'link', 'if', 'the', 'old', 'link', 'has', 'expired']


The question says *stop working*; H02 says *expire*. To a word count those are different words, and so are
*password* and *passwords*.

**BM25** ([BM25 scoring variants](https://bm25s.github.io/)) adds up a score for each question word a document
contains:

$$
\text{BM25}(q, d) = \sum_{t \in q} \text{IDF}(t) \cdot \frac{f(t, d) \cdot (k_1 + 1)}{f(t, d) + k_1 \cdot \left(1 - b + b \cdot \frac{|d|}{\text{avgdl}}\right)}
\qquad
\text{IDF}(t) = \ln\left(1 + \frac{N - n(t) + 0.5}{n(t) + 0.5}\right)
$$

Each symbol and the name it has in the code below:

- $t \in q$: each distinct word of the question, `set(terms(question))`.
- $f(t, d)$: how many times the word appears in the document, `tf`. A word the document lacks adds 0.
- $\text{IDF}(t)$: the inverse document frequency, `idf`. $N$ is the number of documents, `n`, and $n(t)$ the number
  that contain the word, `df`. **Rare words count more.**
- $k_1 = 1.2$: the fraction grows with `tf` but never passes $k_1 + 1$. **Repeats count less and less.**
- $|d|$ and avgdl: the document's length in words, `length`, and the average length, `average`; $b = 0.75$ sets how
  much a long document is marked down. **A match in a short document counts more.**

We choose `k1=1.2` and `b=0.75` for this comparison. They are settings of the scoring rule, rather than learned
probabilities. At `b=0` document length has no effect; at larger `b` a long document needs more occurrences of a word
to earn the same contribution. Increasing `k1` makes repeated occurrences matter more before their benefit flattens.
We keep these settings fixed when moving to the library.

For one contribution, take *the* in H02. The five lengths are 13, 17, 13, 13 and 12 words, so `average=13.6`.
Only H02 contains *the*: `df=1`, `tf=1`, `n=5`.

$$
\text{IDF}(\text{the})=\ln(1+4.5/1.5)=\ln 4\approx1.386
$$
$$
\text{contribution}=1.386\times\frac{1\times2.2}{1+1.2(0.25+0.75\times17/13.6)}\approx1.258
$$

The common product word *atlas* adds about 0.261, giving H02 about 1.519. The arithmetic rewards *the* even though
that word tells us little about link expiry. Tokenization and the corpus determine what counts as rare.
Now express the same sum in Python:


In [4]:
# bm25(): one BM25 score per text for the question, the formula above line by line
def bm25(question, texts, k1=1.2, b=0.75):
    counts = [Counter(terms(text)) for text in texts]
    lengths = [sum(count.values()) for count in counts]
    n, average = len(texts), sum(lengths) / len(texts)
    scores = []
    for count, length in zip(counts, lengths):
        score = 0
        for word in set(terms(question)):
            df = sum(word in other for other in counts)          # documents that contain the word
            idf = math.log(1 + (n - df + 0.5) / (df + 0.5))      # rare word, large weight
            tf = count[word]                                      # times it appears in this document
            score += idf * tf * (k1 + 1) / (tf + k1 * (1 - b + b * length / average))
        scores.append(score)
    return scores

### Keep each score attached to its policy

`counts` is a list of five `Counter` objects. `lengths` has the same five positions, and `zip(counts, lengths)`
keeps a policy's word counts beside its length. Each inner loop adds one distinct question word's contribution;
`Counter` returns zero for an absent word. `scores` is a list of floats in the original policy order.

The next cell pairs those scores with the policies using `zip(scores, policies)`, then sorts the pairs by
`-pair[0]`, the negative score. That puts large scores first without detaching them from their policies.
The set intersection `&` finds shared words; sorting those words makes the printed explanation easy to compare.


In [5]:
# scores: the five BM25 scores; each line also shows the question words the policy contains
scores = bm25(recovery, [policy["text"] for policy in policies])
for score, policy in sorted(zip(scores, policies), key=lambda pair: -pair[0]):
    shared = sorted(set(terms(recovery)) & set(terms(policy["text"])))
    print(f"{policy['source_id']}  {score:.3f}  shares {shared}")

H01  1.705  shares ['atlas', 'email']
H02  1.519  shares ['atlas', 'the']
H04  1.412  shares ['password']
H05  0.302  shares ['atlas']
H03  0.293  shares ['atlas']


BM25 ranks H01 first too: it shares *atlas* and *email*. H02 is second only because it contains *the*, which no other
policy contains, so in five documents it counts as rare. The words that carry the meaning, *stop working* and
*expire*, do not match.

### Check `bm25s` against the hand version

[`bm25s`](https://bm25s.github.io/) is a maintained, fast BM25 package. Its defaults drop English stop words
such as *the*, ignore one-letter words and use `k1=1.5`, so set it up like the hand version and compare the scores.
`bm25s.tokenize()` splits texts into words with the pattern you give it; `BM25(...).index()` counts them;
`get_scores()` scores every document for one question's words:

`get_scores(question_words)` returns one score per indexed policy, in index order, rather than a ranked list.
The loop keeps each score attached to the matching hand score and policy. Tokenization does not replace words
with synonyms or stem *passwords* into *password* here; both implementations must receive the same word choices
for this comparison to isolate the scoring formula.


In [6]:
# index: bm25s over the five policies, set up like bm25(); each line prints both scores and their ratio
import bm25s

WORDS = r"(?u)\b\w+\b"                       # every word, as terms() keeps them
tokens = bm25s.tokenize([policy["text"] for policy in policies], token_pattern=WORDS, stopwords=None,
                        show_progress=False)
index = bm25s.BM25(k1=1.2, b=0.75)
index.index(tokens, show_progress=False)

question_words = bm25s.tokenize([recovery], token_pattern=WORDS, stopwords=None, show_progress=False,
                                return_ids=False)[0]
for policy, mine, theirs in zip(policies, scores, index.get_scores(question_words)):
    print(f"{policy['source_id']}  by hand {mine:.3f}  bm25s {theirs:.3f}  ratio {mine / theirs:.2f}")

H01  by hand 1.705  bm25s 0.775  ratio 2.20
H02  by hand 1.519  bm25s 0.690  ratio 2.20
H03  by hand 0.293  bm25s 0.133  ratio 2.20
H04  by hand 1.412  bm25s 0.642  ratio 2.20
H05  by hand 0.302  bm25s 0.137  ratio 2.20


Every ratio is 2.20, `k1 + 1`: the same order on a different scale. `bm25s` uses the variant of Lucene, the engine
inside Elasticsearch, which leaves $(k_1 + 1)$ out of the numerator. The factor is constant for this fixed `k1`, so dividing every score by it preserves the order. It does not make
BM25 scores comparable with cosine similarity, with another query's scores, or with scores from a different corpus.

### Checkpoint A

| You should see | Our run |
|---|---|
| BM25 by hand | H01 1.705, H02 1.519, H04 1.412, H05 0.302, H03 0.293; H02 shares `['atlas', 'the']` |
| `bm25s` | every ratio 2.20; H01 0.775, H02 0.690 |


---

## Part B: Compare keyword and vector rankings on the same corpus

The MVP question gives keyword matching a useful test: a short exact term can distinguish the requested fact from
other text about the same game. We will inspect that example before running both searches over all 30 questions.

The vector search is **dense retrieval**: every text becomes an embedding, a list of 1,536 numbers that are almost
all non-zero, and texts close in meaning come out close. A keyword search is **sparse** or **lexical retrieval**: a
text is the counts of its words, zero for every word it lacks, and a question matches the documents that share its
exact words. Keyword search finds names, codes and numbers that an embedding can blur, and misses a paraphrase that
uses other words. **Hybrid search** runs both and merges the two lists. Whether that helps on these questions is what
Parts B to D measure.

The keyword ranking is **BM25**, which scores a document $d$ for a question $q$ by the words they share:

$$
\text{BM25}(q, d) = \sum_{t \in q} \text{IDF}(t) \cdot \frac{f(t, d) \cdot (k_1 + 1)}{f(t, d) + k_1 \cdot \left(1 - b + b \cdot \frac{|d|}{\text{avgdl}}\right)}
\qquad
\text{IDF}(t) = \ln\left(1 + \frac{N - n(t) + 0.5}{n(t) + 0.5}\right)
$$

- $t \in q$: each distinct word of the question; the score is a sum over them, and a word the document lacks adds 0.
- $f(t, d)$: the **term frequency**, how many times the word appears in the document.
- $\text{IDF}(t)$: the **inverse document frequency**. Of $N$ documents, $n(t)$ contain the word; a word in few
  documents gets a large weight, a word in nearly all of them a weight near 0.
- $k_1 = 1.2$: the fraction grows with $f(t, d)$ but never passes $k_1 + 1$, so a tenth repeat adds less than a
  second.
- $|d|$ and avgdl: the document's length in words and the average length; $b = 0.75$ sets how much a long document
  is marked down.

For example, *MVP* occurs in two of the 277 chunks. Matching that term earns more weight than matching a word
found almost everywhere. A match still does not tell us whether the passage names the game's MVP or a season's MVP.
BM25 scores are ranking weights for this query and index, with no probability interpretation. Part A (self-study) computes this formula by hand on the five policies. Here the
[`bm25s`](https://bm25s.github.io/) package computes it on the whole corpus.

First the baseline. `shared/lc_pipeline.py` holds the pipeline Lab 05 rebuilt with LangChain, and Lab 06 Part D
prints its four functions: `build_store()` embeds the records into an in-memory vector store, `retrieve()` returns the
`k` closest with their cosine scores, best first, `evidence()` writes hits as `[ID] text` lines, and `answer()` asks
the chat model for a cited answer. This lab uses the first three:

In Lab 05, `OpenAIEmbeddings.embed_query()` produced a question vector, `embed_documents()` produced text
vectors, and the store's `similarity_search_with_score()` returned `(Document, score)` pairs. `build_store()` and
`retrieve()` below reuse those operations and settings. No new retriever object is hidden behind this helper.
We explicitly request ten candidates later; the helper's default is three.


In [7]:
# corpus: the 277 records (272 chunks, five policies); store: their embeddings; questions: the 30 development questions
import json
from shared import lc_pipeline

corpus = lc_pipeline.load_corpus()
store = lc_pipeline.build_store(lc_pipeline.to_documents(corpus), settings)
questions = json.loads((ROOT / "data" / "development-questions.json").read_text(encoding="utf-8"))
print(len(corpus), "records in the store |", len(questions), "questions")
print("question 5:", questions[4]["question"], "| known_support:", questions[4]["known_support"])

277 records in the store | 30 questions
question 5: Who was the Super Bowl 50 MVP? | known_support: ['S00-P02']


277 records, the 272 SQuAD chunks plus the five policies, one embedding each. Each question carries
`known_support`, the paragraph it was written from: question 5 was written from paragraph `S00-P02`, so any chunk of
that paragraph (`S00-P02-C00`, ...) counts as a known source.

Now a BM25 index of the same 277 texts. Three `bm25s` calls:

- `bm25s.tokenize(texts, token_pattern=WORDS, stopwords=None)` splits each text into words. Its defaults drop English
  stop words such as *the* and words of one letter; these arguments keep every word, as the formula above counts them.
- `bm25s.BM25(k1=1.2, b=0.75, corpus=ids)` makes the index with the usual values (its default `k1` is 1.5);
  `corpus=` the chunk IDs makes a search return IDs.
- `.index(tokens)` counts the words of every text.

The corpus dictionaries keep `chunk_id` for the returned chunk, `source_id` for its paragraph or policy and
`text` for searching. A LangChain `Document` keeps the same text as `page_content`, ID as `id` and other fields in
`metadata`. The evaluator reads `known_support` afterwards; it is never part of the query sent to either search.
A paragraph label can cover several chunks, so a source hit is a screening measure rather than proof of a complete answer.


In [8]:
# bm25_corpus: a BM25 index of the 277 texts that returns chunk IDs; corpus_tokens.vocab: every distinct word it holds
import bm25s

WORDS = r"(?u)\b\w+\b"                       # every word, one letter or more
bm25_corpus = bm25s.BM25(k1=1.2, b=0.75, corpus=[record["chunk_id"] for record in corpus])
corpus_tokens = bm25s.tokenize([record["text"] for record in corpus], token_pattern=WORDS, stopwords=None,
                               show_progress=False)
bm25_corpus.index(corpus_tokens, show_progress=False)
print("distinct words in the index:", len(corpus_tokens.vocab))

distinct words in the index: 4083


4,083 distinct words across the 277 texts. Each text uses a few dozen of them, so most of its counts are zero: that is
why lexical retrieval is called sparse.

Try both searches on question 5, the one development question the vector search ranks wrong at the top:

> *Who was the Super Bowl 50 MVP?*

`retrieve(store, question, k=3)` gives (Document, cosine score) pairs. `bm25_corpus.retrieve(tokens, k=3)` gives two
arrays, the IDs and their BM25 scores, with one row per question searched; row `[0]` is this question:

Read the candidate text as well as its ID. These excerpts come from the bundled chunks used in the saved example:

| Chunk | Relevant content |
|---|---|
| S00-P00-C00 | Game summary: Denver Broncos defeated Carolina Panthers 24–10; date and stadium. |
| S00-P00-C01 | The fiftieth game's golden anniversary and choice of Arabic numerals in its logo. |
| S00-P01-C00 | Cam Newton was named the NFL Most Valuable Player (MVP) during the regular season. |
| S00-P02-C00 | Denver linebacker Von Miller was named Super Bowl MVP. |

The season's award and the game's award are different facts. The question asks for the game's award.
For one query and `k=3`, both arrays have shape `(1, 3)`: one query row, three ranked columns. `ids[0][j]` belongs
to `bm25_scores[0][j]`. The vector result is a list of three pairs, with the same best-first convention.


In [9]:
# the top 3 of each search for the MVP question, with each search's own score; then the chunks that contain "MVP"
mvp = "Who was the Super Bowl 50 MVP?"
print("vector:", [(document.id, round(score, 3)) for document, score in lc_pipeline.retrieve(store, mvp, k=3)])

ids, bm25_scores = bm25_corpus.retrieve(bm25s.tokenize([mvp], token_pattern=WORDS, stopwords=None, show_progress=False),
                                        k=3, show_progress=False)
print("BM25:  ", [(str(chunk_id), round(float(score), 3)) for chunk_id, score in zip(ids[0], bm25_scores[0])])
print("chunks that contain 'MVP':", [record["chunk_id"] for record in corpus if "MVP" in record["text"]])

vector: [('S00-P00-C00', 0.68), ('S00-P02-C00', 0.675), ('S00-P01-C00', 0.601)]
BM25:   [('S00-P02-C00', 9.805), ('S00-P00-C01', 8.151), ('S00-P00-C00', 7.758)]
chunks that contain 'MVP': ['S00-P01-C00', 'S00-P02-C00']


- The vector search puts the game summary `S00-P00-C00` first and the answer `S00-P02-C00` a close second.
- BM25 puts the answer first. *MVP* is in only two of the 277 chunks, a rare word with a large IDF. The printed *MVP* inventory also includes the season-award chunk, so the term's presence alone cannot select the
  right award. The final order reflects all query words and document lengths.
- The scores are on different scales: cosine similarities of about 0.68, BM25 scores of about 10. Part C comes back to this.

`str(chunk_id)` turns the NumPy string `bm25s` returns into a plain Python string. The same call as a function that
returns the top 10 IDs, the depth Part C fuses and the reranker in Part E reads:

The next function wraps the `tokenize()` and `retrieve()` calls we just used. `_` deliberately discards the score
array: fusion needs ordered IDs. `ids[0]` selects the single query's row, and the comprehension converts each NumPy
string to a Python `str`. Ask for at most the number of indexed records; this index has 277.


In [10]:
# keyword_search(): the IDs of the k chunks with the highest BM25 scores for one question, best first
def keyword_search(question, k=10):
    tokens = bm25s.tokenize([question], token_pattern=WORDS, stopwords=None, show_progress=False)
    ids, _ = bm25_corpus.retrieve(tokens, k=k, show_progress=False)
    return [str(chunk_id) for chunk_id in ids[0]]

print(keyword_search(mvp))

['S00-P02-C00', 'S00-P00-C01', 'S00-P00-C00', 'S00-P01-C00', 'S00-P01-C01', 'S34-P01-C01', 'S01-P00-C00', 'S22-P02-C00', 'S06-P01-C01', 'S38-P01-C00']


To compare the two searches on all 30 questions, keep for each question the **rank of the first known source**: the
position, from 1, of the first chunk whose source is in `known_support`, or `None` when none of the ten is. Lab 01's
hit@3 then comes from those ranks, and so does hit@1:

$$
\text{hit@}k = \frac{\text{questions whose first known source is at rank } k \text{ or better}}{\text{all questions}}
$$

`src` maps each chunk ID to its paragraph or policy ID. `first_hit()` is Lab 06's `first_rank()` for lists of chunk
IDs: `src` maps each chunk to its paragraph first. Try `first_hit()` on the MVP question:

`enumerate(ids, start=1)` gives each candidate its rank. `src[chunk_id]` converts a chunk ID such as
`S00-P02-C00` to paragraph `S00-P02`; a policy ID such as H02 maps to itself. The function returns immediately on the
first match. Its `None` means no known source in the supplied list, not that the source is absent from the whole corpus.
We search ten, then test ranks at one and three; those are different cutoffs.


In [11]:
# src: chunk ID -> paragraph or policy ID; first_hit(): the rank of the first chunk from a known source, or None
src = {record["chunk_id"]: record["source_id"] for record in corpus}

def first_hit(ids, question):
    for rank, chunk_id in enumerate(ids, start=1):
        if src[chunk_id] in question["known_support"]:
            return rank
    return None

vector_ids = [document.id for document, _ in lc_pipeline.retrieve(store, mvp, k=10)]
print("vector:", first_hit(vector_ids, questions[4]), "| BM25:", first_hit(keyword_search(mvp), questions[4]))

vector: 2 | BM25: 1


Rank 2 for the vector search, rank 1 for BM25, as the lists above showed. Now both searches on all 30 questions, ten
IDs each:

For each question we retain the ten `(Document, score)` pairs in `dense_hits`. `dense_lists` removes the scores
and Documents but preserves their IDs and order; `bm25_lists` has the keyword order. Every outer list has 30 positions,
with position `i` belonging to `questions[i]`. The rank lists keep that same relationship, including `None` misses.
`zip()` joins by position and stops at the shortest list, so equal lengths and matching question order are assumptions
of this code. A shuffled list of results would make the later comparison meaningless.


In [12]:
# dense_hits: the vector search's top 10 (Document, score) pairs per question; dense_lists and bm25_lists: the IDs only
# dense_ranks and bm25_ranks: the first known-source rank per question, None when it is not in the ten
dense_hits = [lc_pipeline.retrieve(store, q["question"], k=10) for q in questions]
dense_lists = [[document.id for document, _ in found] for found in dense_hits]
bm25_lists = [keyword_search(q["question"]) for q in questions]

dense_ranks = [first_hit(ids, q) for ids, q in zip(dense_lists, questions)]
bm25_ranks = [first_hit(ids, q) for ids, q in zip(bm25_lists, questions)]
print("vector ranks:", dense_ranks)
print("BM25 ranks:  ", bm25_ranks)

vector ranks: [1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
BM25 ranks:   [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 8, 1, 1, 1, None, 2, 1, 1, 3, 1, 1, 1, 1, 2, 3, 2, 1, 2]


One rank per question, in the order of the file. `hits(ranks, k)` turns them into True or False per question; the sum
of the Trues is the count of hits:

For example, the MVP's vector rank 2 produces `False` at `k=1` and `True` at `k=3`. A `None` produces `False`
at either cutoff. Python counts `True` as 1 when summing booleans; the printed “of 30” keeps every question in the
denominator. It counts questions with a source hit, rather than relevant chunks retrieved.

To inspect any question's complete candidate texts before trusting the aggregate, use its position in `questions`.
For example, these local statements inspect the already-retrieved MVP candidates without another search:

```python
i = 4
print(questions[i]["question"])
for document, score in dense_hits[i]:
    print(document.id, score, document.page_content)
```

For its keyword candidates, inspect the corpus records whose `chunk_id` appears in `bm25_lists[i]`, in that list's
order. The query, candidate text and position belong together; a rank list alone cannot establish support.


In [13]:
# hits(): True for each question whose first known source is at rank k or better; the sums are hit@1 and hit@3
def hits(ranks, k):
    return [rank is not None and rank <= k for rank in ranks]

for name, ranks in [("vector", dense_ranks), ("BM25", bm25_ranks)]:
    print(f"{name:6}  hit@1 {sum(hits(ranks, 1))} of 30 | hit@3 {sum(hits(ranks, 3))} of 30")

vector  hit@1 29 of 30 | hit@3 30 of 30
BM25    hit@1 22 of 30 | hit@3 28 of 30


The totals say BM25 is worse. The questions where the two ranks differ say how:

In [14]:
# every question whose first known-source rank differs between the two searches
for q, dense_rank, bm25_rank in zip(questions, dense_ranks, bm25_ranks):
    if dense_rank != bm25_rank:
        print(f"vector {dense_rank} | BM25 {bm25_rank} | {q['question'].strip()}")

vector 2 | BM25 1 | Who was the Super Bowl 50 MVP?
vector 1 | BM25 8 | In what country is Normandy located?
vector 1 | BM25 None | What is the original meaning of the word Norman?
vector 1 | BM25 2 | When was the Latin version of the word Norman first recorded?
vector 1 | BM25 3 | In what year did Nikola Tesla emigrate to the United States?
vector 1 | BM25 2 | By what main attribute are computational problems classified utilizing computational complexity theory?
vector 1 | BM25 3 | What measure of a computational problem broadly defines the inherent difficulty of the solution?
vector 1 | BM25 2 | What method is used to intuitively assess or quantify the amount of resources required to solve a computational problem?
vector 1 | BM25 2 | What field of computer science analyzes the resource requirements of a specific algorithm isolated unto itself within a given problem?


BM25 alone ranks the MVP answer higher, and every other question that moved went down; for the Norman question the
answering chunk is not in its top ten at all (`None`). These questions were written from their paragraphs, so they
share many words with them, and the vector search already ranks a known source first for 29 of 30. Hybrid search
hopes to keep the vector search's results and add BM25's one win.

The vector ranking uses similarity between text representations; lexical ranking uses matching tokens. Neither
checks facts. Here we measured which returned sources match an answer key, and inspected the MVP passage to check
what the different source ranks meant. Part C tests whether merging the lists preserves the useful results.

### Checkpoint B

| You should see | Our run |
|---|---|
| Index | 277 records; 4,083 distinct words |
| The MVP question | vector `S00-P00-C00`, `S00-P02-C00`, `S00-P01-C00`; BM25 `S00-P02-C00` first (9.805); ranks 2 and 1 |
| 30 questions | vector hit@1 29, hit@3 30; BM25 hit@1 22, hit@3 28; nine questions with different ranks |


---

## Part C: Combine rankings and trace why fusion can lower an answer

Part C merges the vector list and the BM25 list into one ranking.

Adding the raw scores would give their scales unintended influence: in Part B, BM25 scored the MVP answer about 10 and the vector search about 0.68, so
BM25's larger scale would dominate the sum. A calibrated weighted score is another design; we use ranks here to avoid that calibration. **Reciprocal rank fusion** (RRF, [Cormack and others,
2009](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf)) uses only the ranks:

$$
\text{RRF}(d) = \sum_{L} \frac{1}{c + \text{rank}_L(d)}
$$

- $L$: each ranked list, here the vector list and the BM25 list.
- $\text{rank}_L(d)$: the chunk's position in that list, 1 for the first. A list that does not hold the chunk adds
  nothing.
- $c = 60$: a constant from that paper. It keeps the ranks close: first place gives $1/61$, tenth $1/70$, so a chunk
  first in one list scores less than a chunk anywhere in both lists' top ten.

`c` and each list's depth (10 here) are fixed before measuring: tuning them until the 30 questions look better would
measure the tuning.

Try it on a keyword list B, A and a vector list A, C, B:

The depth is how many candidates each search contributes, rather than how many we will pass to an answering
model. Two top-ten lists can contain up to 20 distinct IDs; fusion returns their union in a new order. Smaller `c`
makes the first few positions matter more; larger `c` makes their contributions closer. With `c=60`, even two
rank-ten contributions, `2/70`, exceed one rank-one contribution, `1/61`. Agreement can therefore displace a source
that only one search finds. We give both searches equal weight.


In [15]:
# fused: each item's RRF score, built list by list; printed best first
keyword = ["B", "A"]
vector = ["A", "C", "B"]
fused = {}
for ranked in (keyword, vector):
    for rank, item in enumerate(ranked, start=1):
        fused[item] = fused.get(item, 0) + 1 / (60 + rank)
for item, score in sorted(fused.items(), key=lambda pair: -pair[1]):
    print(item, f"{score:.6f}")

A 0.032522
B 0.032266
C 0.016129


A is second in one list and first in the other: $1/62 + 1/61 = 0.032522$. B is first once but third in the other, so
it scores a little less; C is in one list only. **Fusion rewards agreement.** The same loop as a function that returns
the IDs, best first:

`fused.get(item, 0)` starts an unseen ID at zero, then accumulates its contribution from each list. The dictionary
has one key per distinct ID, so a candidate shared by both lists is returned once. `sorted(..., key=...)` uses the
negative total to order high scores first. Exact ties keep insertion order in this Python implementation; list order
can therefore decide a tie. RRF supplies no judgement about whether A, B or C actually answers the query.


In [16]:
# rrf(): the IDs of every list, ordered by their summed 1 / (c + rank)
def rrf(lists, c=60):
    scores = {}
    for ranked in lists:
        for rank, chunk_id in enumerate(ranked, start=1):
            scores[chunk_id] = scores.get(chunk_id, 0) + 1 / (c + rank)
    return sorted(scores, key=lambda chunk_id: -scores[chunk_id])

print(rrf([keyword, vector]))

['A', 'B', 'C']


Part B showed one question BM25 misses entirely:

> *What is the original meaning of the word Norman?*

*Predict first:* the vector search ranks its answer first; where will fusion put it? Fuse its two lists and print where
each of the top five came from:

The candidate passages differ in what they explain:

| Chunk | Content to inspect |
|---|---|
| S02-P01-C00 | The Norman dynasty's political, cultural and military impact; the Norman language. |
| S02-P00-C00 | The Norman people, Normandy in France, and “Norman” from “Norseman”. |
| S02-P01-C02 | Norman conquests and cultural influence. |
| S02-P02-C00 | Name origins, including “Northman” and “Norseman, Viking”; the labelled source. |
| S26-P00-C00 | The original meaning of *pharmacy*, a different word. |

The first two sources about the Normans are on topic, but on-topic text and the labelled answer source are different
checks. S02-P00-C00 even offers a related etymology: `known_support` records where this question's reference came
from, rather than every possible supporting passage. The following ranks measure that supplied label.


In [17]:
# the Norman question's fused top five, each with its rank in both lists (None when missing) and its RRF score
norman = next(i for i, q in enumerate(questions) if "original meaning" in q["question"])
lists = [dense_lists[norman], bm25_lists[norman]]

for chunk_id in rrf(lists)[:5]:
    ranks = [ids.index(chunk_id) + 1 if chunk_id in ids else None for ids in lists]
    score = sum(1 / (60 + rank) for rank in ranks if rank)
    known = "  <- known source" if src[chunk_id] in questions[norman]["known_support"] else ""
    print(f"{chunk_id}  vector {ranks[0]} | BM25 {ranks[1]} | RRF {score:.4f}{known}")

S02-P01-C00  vector 3 | BM25 3 | RRF 0.0317
S02-P00-C00  vector 2 | BM25 5 | RRF 0.0315
S02-P01-C02  vector 6 | BM25 6 | RRF 0.0303
S02-P02-C00  vector 1 | BM25 None | RRF 0.0164  <- known source
S26-P00-C00  vector None | BM25 1 | RRF 0.0164


The known source `S02-P02-C00` is first in the vector list and missing from the BM25 list, so it gets one
contribution, $1/61 = 0.0164$. Three chunks in both lists' top six score about 0.030 to 0.032 and push it to fourth.
The pharmacy passage also receives one rank-one contribution, but for a different query word's meaning.
Neither its RRF score nor the agreement on Norman-history passages establishes factual support for this question.

For this query, agreement pushes the labelled source outside the top three. A fusion method can help some queries
and harm others; the next comparison keeps the before/after outcome attached to each question.

### Checkpoint C

| You should see | Our run |
|---|---|
| The example | A 0.032522, B 0.032266, C 0.016129; `rrf()` gives `['A', 'B', 'C']` |
| The Norman question | `S02-P02-C00` fourth with 0.0164, behind 0.0317, 0.0315 and 0.0303 |


---

## Part D: Compare hybrid search with the baseline, question by question

Part D measures the fused list against the vector list on the same 30 questions.

*Predict first:* the vector search ranks a known source first for 29 of 30 questions; how many will hybrid search
rank first?

Fuse every question's two lists and count hit@1 and hit@3:

In [18]:
# rrf_lists: the fused list per question; rrf_ranks: its first known-source rank; the totals beside the vector search's
rrf_lists = [rrf([dense, keyword]) for dense, keyword in zip(dense_lists, bm25_lists)]
rrf_ranks = [first_hit(ids, q) for ids, q in zip(rrf_lists, questions)]
for name, ranks in [("vector", dense_ranks), ("hybrid", rrf_ranks)]:
    print(f"{name:6}  hit@1 {sum(hits(ranks, 1))} of 30 | hit@3 {sum(hits(ranks, 3))} of 30")

vector  hit@1 29 of 30 | hit@3 30 of 30
hybrid  hit@1 27 of 30 | hit@3 29 of 30


Two totals hide which questions moved: 29 against 27 could be two lost, or three lost and one gained. A **paired
comparison** runs both versions on the same questions and puts each question in one of four cells:

| | hit after the change | miss after the change |
|---|---|---|
| **hit before** | both | lost |
| **miss before** | gained | neither |

*Gained* and *lost* determine the change in hit count. *Both* and *neither* still matter: they show retained
successes and unresolved failures. `paired()` sorts the questions at one `k`; try it at hit@1:

The inputs to `paired()` are two equal-length boolean lists in the same question order. Its `both` and `neither`
fields are counts; `gained` and `lost` are lists of zero-based positions, so we can inspect the changed questions.
For the printed result, `26 + 0 + 1 + 3 = 30`. The baseline has `26 + 3 = 29` hits; hybrid has `26 + 1 = 27`.
That accounts for the totals while retaining the four questions responsible for the difference.


In [19]:
# paired(): how many questions passed both ways or neither, and the positions of the questions gained and lost
def paired(before, after):
    pairs = list(zip(before, after))
    return {"both": sum(b and a for b, a in pairs), "neither": sum(not b and not a for b, a in pairs),
            "gained": [i for i, (b, a) in enumerate(pairs) if a and not b],
            "lost": [i for i, (b, a) in enumerate(pairs) if b and not a]}

paired(hits(dense_ranks, 1), hits(rrf_ranks, 1))

{'both': 26, 'neither': 0, 'gained': [4], 'lost': [12, 16, 26]}

`gained` and `lost` hold positions in `questions`, counted from 0: `[4]` is question 5, the MVP question. `show()`
prints the counts at hit@3 and hit@1 and names every question that moved, with its rank before and after:

In [20]:
# show(): the paired counts at hit@3 and hit@1, then each question gained or lost with its two ranks
def show(before_ranks, after_ranks, after_name):
    for k in (3, 1):
        table = paired(hits(before_ranks, k), hits(after_ranks, k))
        print(f"hit@{k}: both {table['both']} | gained {len(table['gained'])} | lost {len(table['lost'])}"
              f" | neither {table['neither']}")
        for i in table["gained"] + table["lost"]:
            moved = "gained" if i in table["gained"] else "lost  "
            print(f"   {moved} vector rank {before_ranks[i]} | {after_name} rank {after_ranks[i]} | {questions[i]['question'].strip()}")

show(dense_ranks, rrf_ranks, "hybrid")

hit@3: both 29 | gained 0 | lost 1 | neither 0
   lost   vector rank 1 | hybrid rank 4 | What is the original meaning of the word Norman?
hit@1: both 26 | gained 1 | lost 3 | neither 0
   gained vector rank 2 | hybrid rank 1 | Who was the Super Bowl 50 MVP?
   lost   vector rank 1 | hybrid rank 2 | In what country is Normandy located?
   lost   vector rank 1 | hybrid rank 4 | What is the original meaning of the word Norman?
   lost   vector rank 1 | hybrid rank 3 | What measure of a computational problem broadly defines the inherent difficulty of the solution?


At hit@3 hybrid search keeps 29 questions and loses the Norman question. At hit@1 it gains the MVP question, BM25's
one win, and loses three questions the vector search had right.

Is a split of one gained and three lost evidence, or chance? The [**sign test**](https://en.wikipedia.org/wiki/Sign_test)
uses a null model in which gains and losses are equally likely among the changed questions. Assuming those
question outcomes are independent, its two-sided p-value is the probability, under that model, of a split at least
as uneven in either direction. It is not the probability that the change makes no difference:

$$
p = \min\left(1,\; 2 \sum_{i=0}^{m} \binom{n}{i} \left(\frac{1}{2}\right)^{n}\right)
\qquad n = \text{gained} + \text{lost}, \quad m = \min(\text{gained}, \text{lost})
$$

- $n$: the questions that moved. Questions that passed or failed both ways are ties and are left out.
- $\binom{n}{i} (1/2)^n$: the chance that exactly $i$ of $n$ fair coin flips land on one side (the binomial
  distribution). `math.comb(n, i)` is $\binom{n}{i}$, the number of ways to choose $i$ of $n$.
- $\sum_{i=0}^{m}$: the chance of a side as small as ours or smaller; the 2 counts an equally uneven split the other
  way; $\min(1, \dots)$ keeps it a probability.

For 1 gained and 3 lost: $n = 4$ and $m = 1$. Of the $2^4 = 16$ equally likely gain/loss sequences, $\binom{4}{0} + \binom{4}{1} = 1 + 4$
have one gain or none, so the sum is $5/16 = 0.3125$, doubled $0.625$. A small p-value (below 0.05 by
convention) would say the split is unlikely under "no difference". On paired pass and fail results the same test is
called the **exact McNemar test**.


In [21]:
# sign_test(): the two-sided p-value for a split of gains and losses; printed at hit@3 and hit@1
from math import comb

def sign_test(gains, losses):
    n = gains + losses
    tail = sum(comb(n, i) for i in range(min(gains, losses) + 1)) / 2 ** n
    return min(1.0, 2 * tail)

for k in (3, 1):
    table = paired(hits(dense_ranks, k), hits(rrf_ranks, k))
    gains, losses = len(table["gained"]), len(table["lost"])
    print(f"hit@{k}: {gains} gained, {losses} lost | p = {sign_test(gains, losses):.3f}")

hit@3: 0 gained, 1 lost | p = 1.000
hit@1: 1 gained, 3 lost | p = 0.625


At hit@1 only four questions change: one gain and three losses give p = 0.625. At hit@3 just one changes, giving
p = 1.000. Neither result rejects the equal-gain/loss null model at the conventional 0.05 threshold. A large p-value
does not establish equivalence or tell us the probability that chance caused these results.

The 30 questions share five source articles. Related questions can have related errors, so the test's independence
assumption is doubtful; these p-values are an arithmetic demonstration, not a reliable generalisation claim.
We also inspected these development examples while designing the pipeline. A fresh customer-question set would be
needed for a stronger comparison.

We can still make a local engineering decision: the measured hybrid configuration adds an index and lowers hit@3
on this set. Keep the vector baseline while collecting broader evidence. This does not rule out other tokenizers,
weights, depths or corpora. Those would be new configurations to evaluate.

### Checkpoint D

| You should see | Our run |
|---|---|
| Totals | vector hit@1 29, hit@3 30; hybrid hit@1 27, hit@3 29 |
| Paired, hit@3 | both 29, gained 0, lost 1 (Norman) |
| Paired, hit@1 | both 26, gained 1 (MVP), lost 3 (Normandy, Norman, the computational measure) |
| Sign test | hit@3 p = 1.000; hit@1 p = 0.625 |


---

## Part E: Let a chat model reorder the recovery question’s candidates

Part E adds a second stage after the search: a **reranker** reads the question with each candidate and reorders the
candidates. Searching wide and then reranking a short list is called **two-stage retrieval**: the first stage searches a large collection for a manageable candidate set; the second reads that set more closely
and tries to move answering passages earlier. A chunk missing from the
candidates stays missing.

One common reranking option is a **cross-encoder**, a model that reads one question and one passage together
and scores the pair (for example [Cohere Rerank](https://docs.cohere.com/docs/rerank-overview)), behind a vendor API
or a model download. With only a chat API, you ask the chat model to rank the whole list in one call: **listwise
reranking**, as in [RankGPT](https://arxiv.org/abs/2304.09542).

The development questions share many words with their paragraphs. Customers word questions their own way; Lab 06
wrote this one (its case G02), and its answer is the Atlas policy H02, *Atlas reset links expire after 30 minutes*:

> *When does the password recovery email stop working for Atlas?*

Where does H02 sit in each list from Parts B and C? `candidates` keeps the vector search's top 10 for the reranker:


In [22]:
# candidates: the vector search's top 10 (Document, score) pairs; H02's rank in the vector, BM25 and fused lists
import time

recovery = "When does the password recovery email stop working for Atlas?"
candidates = lc_pipeline.retrieve(store, recovery, k=10)
vector_ids = [document.id for document, _ in candidates]
bm25_ids = keyword_search(recovery)
for name, ids in [("vector", vector_ids), ("BM25", bm25_ids), ("hybrid", rrf([vector_ids, bm25_ids]))]:
    print(f"{name:6}  H02 at rank {ids.index('H02') + 1} | top three {ids[:3]}")

vector  H02 at rank 3 | top three ['H01', 'H04', 'H02']
BM25    H02 at rank 6 | top three ['H01', 'H04', 'S11-P00-C00']
hybrid  H02 at rank 4 | top three ['H01', 'H04', 'H05']


H02 is third in the vector list, sixth in BM25's and fourth after fusion: the question says *stop working* and the
policy says *expire*. H01, *Atlas passwords can be reset by email*, shares *atlas* and *email* with the question and
comes first everywhere.

*Predict first:* given the ten candidates to read, will a chat model put H02 first?

The reranker is the chat model `RERANK_MODEL`, asked for a reply in a fixed shape:

- `Ranking` is a Pydantic class with one field, `ranking`, the IDs in order.
- `RULES` is the system message: rank by how directly each passage answers, every ID once, best first.
- `lc_chat_model("RERANK_MODEL", settings)` is the LangChain chat model named in `models.env`.
- `.with_structured_output(Ranking, method=..., include_raw=True)` asks for JSON in that shape (a strict schema or
  plain JSON, as `JSON_MODE` says). The reply becomes a dictionary: `"parsed"` holds a `Ranking`, `"raw"` the model's
  message with the token counts.

For a small two-candidate illustration, a reply could be `{"ranking": ["H02", "H01"]}`. The strings identify
passages; their positions express the model's preference. No relevance score or factual answer is returned.
The actual call below receives ten candidates and must return all ten exactly once.

Lab 06 used `with_structured_output()` for cited answers. We now call the same public method directly with a smaller
schema. `lc_chat_model()` reuses the configured `ChatOpenAI` construction shown in Lab 05; the new behaviour is the
ranking schema and prompt, not a different model client. `include_raw=True` also returns parsing information;
this demonstration expects parsing to succeed, so its direct `.ranking` access is not an error-recovery route.


In [23]:
# reranker: the chat model that returns {"parsed": Ranking, "raw": the message}
from pydantic import BaseModel
from shared.settings import lc_chat_model

class Ranking(BaseModel):
    ranking: list[str]

RULES = ("Rank the passages by how directly they answer the question."
         ' Reply in JSON shaped like {"ranking": ["ID", ...]}, with every passage ID once, best first.')
method = "json_schema" if settings["JSON_MODE"] == "json_schema" else "json_mode"
reranker = lc_chat_model("RERANK_MODEL", settings).with_structured_output(Ranking, method=method, include_raw=True)
print("rerank model:", settings["RERANK_MODEL"], "| JSON_MODE:", settings["JSON_MODE"])

rerank model: gpt-4o-mini | JSON_MODE: json_schema


The user message holds the question and the ten candidates as `[ID] text` lines, the format the answer step reads,
from `evidence()`. Print its start before it is sent:

The output is a 420-character preview, not the complete request. The full `user` string includes all ten
candidate texts. The five short policies near its start are the same H01–H05 texts inspected in Part A: H01 supplies
a reset method, H02 an Atlas duration, H04 a Boreal duration, H03 report permissions and H05 billing. If you skipped
Part A, the visible H01/H04/H02 lines below already show why the product and requested fact both matter.
The remaining candidates are corpus chunks; `lc_pipeline.evidence(candidates)` keeps their exact IDs and text.
The reranker sees no `known_support`, expected rank or reference answer.

To inspect the complete input locally before sending it, evaluate `print(user)` after the next cell. That prints
all ten `[ID] text` lines already held in memory; it makes no additional API call. The preview alone cannot show
whether a later candidate contains distracting text or an instruction.


In [24]:
# user: the question and the ten candidates as "[ID] text" lines, one message
user = f"Question: {recovery}\nPassages:\n" + "\n".join(lc_pipeline.evidence(candidates))
print(user[:420], "...")

Question: When does the password recovery email stop working for Atlas?
Passages:
[H01] Atlas passwords can be reset by email. Temporary accounts require a support agent.
[H04] Boreal reset links expire after 10 minutes. Password reset uses a mobile app.
[H02] Atlas reset links expire after 30 minutes. Request a new link if the old link has expired.
[H05] Atlas invoices are sent monthly. Workspace owners can download ...


One call. `reply["parsed"].ranking` is the list of IDs; `usage_metadata` on the raw message holds the tokens billed:

In [25]:
# order: the IDs as the reranker ranked them; the call's tokens and seconds
start = time.perf_counter()
reply = reranker.invoke([("system", RULES), ("user", user)])
seconds = time.perf_counter() - start

order = reply["parsed"].ranking
usage = reply["raw"].usage_metadata
print("vector:  ", vector_ids[:5])
print("reranked:", order[:5])
print(f"tokens: {usage['input_tokens']} in, {usage['output_tokens']} out | {seconds:.2f} s")

vector:   ['H01', 'H04', 'H02', 'H05', 'H03']
reranked: ['H02', 'H01', 'H04', 'H05', 'H03']
tokens: 561 in, 54 out | 1.16 s


H02 moves from vector rank 3 to reranked rank 1. Its sentence “Atlas reset links expire after 30 minutes” answers
the recovery question; H01 describes the method and H04 describes Boreal. Reading those texts lets us judge this
example, beyond observing a changed ID. The model's reason is not returned, so we cannot infer its internal reasoning
from the order alone.

This call used 561 input tokens, 54 output tokens and 1.16 seconds. It demonstrates a useful reordering on one
question. Repeated successes on this same wording would show repeatability for that case, not success on additional
independent questions. Part F tests all 30 development questions; Part G considers five policy questions separately.
The ID check immediately after the checkpoint tests a different property from ranking quality.

### Checkpoint E

| You should see | Our run |
|---|---|
| H02 in each list | vector rank 3, BM25 rank 6, hybrid rank 4; `H01` first in all three |
| Reranked | `['H02', 'H01', 'H04', 'H05', 'H03']`; 561 tokens in, 54 out, 1.16 s |


### Verify that the reply is a permutation of the candidates (self-study)

The schema asks for a list of strings. The prompt asks for every candidate once; our application must still check
that relationship. Sorting both lists ignores their order while preserving the number of occurrences of each ID.
Equal sorted lists mean exactly the same IDs occur exactly as often.

For candidates `["H01", "H02"]`, `["H02", "H01"]` passes. `["H02"]` omits an ID; `["H02", "H02"]` duplicates one;
`["H02", "H99"]` substitutes an outsider. All three fail, even though all have a list-of-strings shape.
A set comparison alone would miss duplicate occurrences when the distinct IDs are unchanged.

A valid candidate permutation can still rank Boreal's H04 first for this Atlas question. This check establishes the
ID relationship only. It neither verifies factual relevance nor protects against instructions inside a passage.
Part F applies it to every call and uses the vector order when it fails:


In [26]:
# True when the reply holds each candidate ID once and nothing else
print("exactly the candidate IDs:", sorted(order) == sorted(vector_ids))

exactly the candidate IDs: True


`True` means all ten candidates are present exactly once in this reply. It does not mean H02 is the best answer;
that judgement came from reading the question and text. Keep both checks conceptually separate when the function
below combines generation and validation.


---

## Part F: Compare validated rerankings and measure their added cost (self-study)

Part F reranks every development question, checks every reply, and prices the new stage.

Part E's steps go into one function. It returns the order to use, the reply's order, whether the reply passed the ID
check, and the call's tokens and seconds:

`found` is a list of `(Document, cosine score)` pairs. The message uses each Document's ID and text, not its
cosine score. The returned dictionary keeps these fields separate:

| Field | Meaning |
|---|---|
| `returned` | IDs the model wrote, in its proposed order |
| `accepted` | Whether those IDs are exactly the candidates, each once |
| `order` | Proposed order if accepted; original vector order otherwise |
| `input_tokens`, `output_tokens` | Usage reported for this call, including rejected replies |
| `seconds` | Wall-clock time around the model invocation |

The list comprehension repeats that same operation for the 30 aligned `(question, hits)` pairs. `known_support`
stays with the evaluator. We measure the combined reranker-and-fallback policy, since that is the order the answer
step would receive.


In [27]:
# rerank(): one reranker call on a question's (Document, score) hits; keeps the search order when the IDs are wrong
def rerank(question, found):
    ids = [document.id for document, _ in found]
    user = f"Question: {question}\nPassages:\n" + "\n".join(lc_pipeline.evidence(found))
    start = time.perf_counter()
    reply = reranker.invoke([("system", RULES), ("user", user)])
    seconds = time.perf_counter() - start
    returned = reply["parsed"].ranking
    accepted = sorted(returned) == sorted(ids)
    usage = reply["raw"].usage_metadata
    return {"order": returned if accepted else ids, "returned": returned, "accepted": accepted,
            "input_tokens": usage["input_tokens"], "output_tokens": usage["output_tokens"], "seconds": seconds}

reranked = [rerank(q["question"], found) for q, found in zip(questions, dense_hits)]
print("replies that passed the ID check:", sum(r["accepted"] for r in reranked), "of 30")

replies that passed the ID check: 28 of 30


### What happens step by step

1. `ids` are the candidates in the search's order; `user` is Part E's message for this question.
2. One reranker call, timed.
3. `returned` is the reply's order; `accepted` is Part E's ID check.
4. `order` is the reply's order when it passed, the search's order when it did not, so a reply with invalid IDs is not used to select evidence. A valid but unhelpful ranking can still be used.

Print every reply that failed the check, with what was wrong:

The diagnostics below use set differences for missing and extra IDs, then count occurrences to find repeats.
An extra ID can be a real corpus ID and still be an invalid candidate for this question. Rejecting a reply also
leaves its token and time cost in the ledger.


In [28]:
# for each rejected reply: the candidates it left out, the IDs that were not candidates, and any repeated ID
for q, ids, r in zip(questions, dense_lists, reranked):
    if not r["accepted"]:
        returned = r["returned"]
        print(q["question"].strip())
        print("   left out:", sorted(set(ids) - set(returned)),
              "| not a candidate:", sorted(set(returned) - set(ids)),
              "| repeated:", sorted({i for i in returned if returned.count(i) > 1}))

When was the Latin version of the word Norman first recorded?
   left out: ['S02-P01-C01'] | not a candidate: [] | repeated: []
Were Tesla's experiments high-voltage or low-voltage?
   left out: [] | not a candidate: ['S03-P00-C01'] | repeated: []


Two replies of 30 failed in our run. For the second Norman question the reply left out one candidate. For the Tesla question it named `S03-P00-C01`, an ID that is not in the corpus at all: Tesla's first paragraph has one chunk, `S03-P00-C00`, and the model wrote a second in the same pattern. Which replies fail changes from run to run: 0 to 3 of 30 per pass across our runs.

Now fix one query and its candidates and repeat the model call:

> *Over how many species of trees can be found in the Saxon Garden?*

The candidate list comes from this question's `dense_hits[saxon]`, not a new search in each repeat. Its known
source is Warsaw paragraph S01-P01, whose garden description holds the species count. A model-written Norman ID
such as S02-P01-C00 belongs to a different article. These five calls test variability with fixed inputs; they add
no new questions to the comparison.


In [29]:
# the Saxon Garden question reranked five times with the same ten candidates
saxon = next(i for i, q in enumerate(questions) if "Saxon Garden" in q["question"])
for run in range(1, 6):
    r = rerank(questions[saxon]["question"], dense_hits[saxon])
    print(f"run {run}: passed {r['accepted']} | left out {sorted(set(dense_lists[saxon]) - set(r['returned']))}"
          f" | not a candidate {sorted(set(r['returned']) - set(dense_lists[saxon]))}")

run 1: passed False | left out ['S01-P02-C00'] | not a candidate ['S02-P01-C00']


run 2: passed False | left out ['S01-P02-C00'] | not a candidate ['S02-P01-C00']


run 3: passed True | left out [] | not a candidate []


run 4: passed False | left out ['S01-P02-C00'] | not a candidate ['S02-P01-C00']


run 5: passed False | left out ['S01-P02-C00', 'S16-P00-C02'] | not a candidate ['S02-P00-C02']


Four of the five replies failed: three left out `S01-P02-C00` and named `S02-P01-C00`, and one left out two candidates and named `S02-P00-C02`. Both are real chunks from the Normans article, so a check against the corpus would pass them; check against the candidates. The same question passed the check in the 30-question pass above. **Count failures over several runs, and keep the check on every call.**

Now the paired table, reranked against the vector order, with Part D's `show()` and sign test:

The repeat count has denominator five calls on one selected question. The previous ID-check count has denominator
30 different questions, one call each. Do not pool them into a failure rate for 35 independent questions.
`rerank_ranks` below reads `order`, including any fallback; it does not score an invalid proposed order.


In [30]:
# rerank_ranks: the first known-source rank in each order used; the paired table and the sign test at hit@1
rerank_ranks = [first_hit(r["order"], q) for r, q in zip(reranked, questions)]
show(dense_ranks, rerank_ranks, "reranked")
table = paired(hits(dense_ranks, 1), hits(rerank_ranks, 1))
print(f"sign test at hit@1: p = {sign_test(len(table['gained']), len(table['lost'])):.3f}")

hit@3: both 30 | gained 0 | lost 0 | neither 0
hit@1: both 28 | gained 1 | lost 1 | neither 0
   gained vector rank 2 | reranked rank 1 | Who was the Super Bowl 50 MVP?
   lost   vector rank 1 | reranked rank 2 | What measure of a computational problem broadly defines the inherent difficulty of the solution?
sign test at hit@1: p = 1.000


hit@3 does not move: 30 of 30 both ways. At hit@1 the reranker gains the MVP question and loses the computational measure question: 29 both ways, p = 1.000. Read the lost question: what did the reranker put first, and what does the known source say?

A total of 29 hits either way hides an exchange: one gained question and one lost. The question text below
identifies the loss; the excerpts show what the reranker preferred. They are previews of the two chunks, not their
complete text or a generated answer.


In [31]:
# for each question the reranker lost at hit@1: the chunk it put first and the known source, first 150 characters each
text = {record["chunk_id"]: record["text"] for record in corpus}
for i in table["lost"]:
    first, known = reranked[i]["order"][0], dense_lists[i][dense_ranks[i] - 1]
    print(questions[i]["question"].strip())
    print(f"   reranked first {first}: {text[first][:150]} ...")
    print(f"   known source  {known}: {text[known][:150]} ...")

What measure of a computational problem broadly defines the inherent difficulty of the solution?
   reranked first S04-P00-C00: Computational complexity theory is a branch of the theory of computation in theoretical computer science that focuses on classifying computational pro ...
   known source  S04-P01-C00: A problem is regarded as inherently difficult if its solution requires significant resources, whatever the algorithm used. The theory formalizes this  ...


The reranker chose `S04-P00-C00`, which defines the field and repeats the question's words *inherent difficulty*, over `S04-P01-C00`, which holds the answer: *if its solution requires significant resources*. Shared words can mislead a reranker too.

Last, the cost. Time the two searches on the 30 questions; the reranker's seconds and tokens come from its replies.
The chat prices in `models.env` apply because `RERANK_MODEL` is the chat model here:

$$
\text{dollars per question} = \frac{\text{tokens in} \times \text{input price} + \text{tokens out} \times \text{output price}}{1{,}000{,}000}
$$

The prices are US dollars per million tokens. The last line counts the evidence the answer step reads, the top three
chunks, for comparison:

These are configured prices in US dollars per million tokens, not a live provider price lookup. If the rerank
model or endpoint differs, update the prices to that provider's rates before interpreting the result. This simple
estimate charges every reported input token at the ordinary input rate; it does not model cached-input discounts,
retries, service fees or other billing tiers.

The 30 calls' tokens are summed first, priced, then divided by 30. Multiplying by 1,000 is a linear projection at
the same average usage. It is not a measurement of 1,000 questions. Corpus indexing, query-embedding charges, the
answer-generation call and infrastructure are outside that dollar figure. The search timers measure warm searches
on the existing indexes; reranker time is additional to candidate retrieval.


In [32]:
# milliseconds per question for each stage, the reranker's tokens and dollars, and the answer step's evidence tokens
import tiktoken
from shared.settings import chat_url

n = len(questions)
start = time.perf_counter()
for q in questions:
    lc_pipeline.retrieve(store, q["question"], k=10)
vector_seconds = time.perf_counter() - start
start = time.perf_counter()
for q in questions:
    keyword_search(q["question"])
bm25_seconds = time.perf_counter() - start

tokens_in = sum(r["input_tokens"] for r in reranked)
tokens_out = sum(r["output_tokens"] for r in reranked)
dollars = (tokens_in * float(settings["CHAT_PRICE_INPUT"]) + tokens_out * float(settings["CHAT_PRICE_OUTPUT"])) / 1e6
encoding = tiktoken.get_encoding(settings["TOKEN_ENCODING"])
evidence = sum(len(encoding.encode("\n".join(lc_pipeline.evidence(found[:3])))) for found in dense_hits)

print(f"vector search  {1000 * vector_seconds / n:.1f} ms per question (one embedding call)")
print(f"BM25 search    {1000 * bm25_seconds / n:.1f} ms per question (no call)")
print(f"rerank         {1000 * sum(r['seconds'] for r in reranked) / n:.0f} ms per question | {tokens_in / n:.0f} tokens in,"
      f" {tokens_out / n:.0f} out | {1000 * dollars / n:.2f} US dollars per 1,000 questions")
print(f"answer step reads {evidence / n:.0f} tokens of evidence per question")
print(f"\n{settings['RERANK_MODEL']} at {chat_url(settings)}")

vector search  232.8 ms per question (one embedding call)
BM25 search    0.1 ms per question (no call)
rerank         1617 ms per question | 961 tokens in, 74 out | 0.19 US dollars per 1,000 questions
answer step reads 281 tokens of evidence per question

gpt-4o-mini at https://api.openai.com/v1


### What happens step by step

1. The two loops run each search once per question and time it; the vector search includes its embedding call.
2. `tokens_in` and `tokens_out` add up the reranker's usage over the 30 replies; `dollars` is the formula above.
3. `evidence` counts, with the `TOKEN_ENCODING` tokenizer, the tokens of the top three `[ID] text` lines the answer
   step reads for each question.
4. Every line is divided by `n`, the 30 questions; the dollars are shown per 1,000 questions.

The vector search takes about 230 ms per question, including the embedding call, and BM25 about 0.1 ms. The reranker takes about 1.6 seconds and reads about 960 tokens, more than three times the 280 or so tokens of evidence the answer reads. At about 0.19 US dollars per 1,000 questions it costs little money, but **this pass averages more than a second of added reranking time per query**.

### Checkpoint F

| You should see | Our run |
|---|---|
| ID check | 28 of 30 passed (the second Norman reply left one candidate out; the Tesla reply named `S03-P00-C01`, not in the corpus); the Saxon Garden question failed 4 of 5 repeats |
| Paired, hit@3 and hit@1 | hit@3: both 30; hit@1: gained 1 (MVP), lost 1 (the computational measure), p = 1.000 |
| Cost per question | vector 232.8 ms, BM25 0.1 ms; rerank 1,617 ms, 961 tokens in, 74 out, 0.19 US dollars per 1,000 questions; evidence 281 tokens |

The “evidence” count is only the original vector top three formatted as `[ID] text`, not a measured answer-model
request. It excludes instructions, the question, schema and answer tokens. Comparing it with the reranker's billed
input illustrates prompt size, but the two counts cover different things. A single sequential pass on a small corpus
does not estimate tail latency, concurrent load or index maintenance cost.


---

## Part G: Inspect policy questions before choosing a retrieval change (self-study)

Part G tests the two changes on questions worded unlike their answers, then writes the decision.

The development questions were written from their paragraphs and share many words with them, so the vector search
already ranks a known source first for 29 of 30. Customers word questions their own way. Five Atlas questions, each
with the policy that answers it: the recovery question from Lab 06 and four from Lab 04:

Before the loop, read what each question asks the candidates to distinguish:

| Question | Known policy and the distinction it supplies |
|---|---|
| When does the password recovery email stop working for Atlas? | H02: 30 minutes, rather than H01's method or H04's Boreal duration |
| How long does an Atlas reset link remain valid? | H02: the same duration, with wording closer to the policy |
| Who can export all reports in Atlas? | H03: only workspace owners |
| Can temporary Atlas accounts reset by email? | H01: temporary accounts require a support agent |
| Do I need the mobile app to reset my Atlas password? | H01: Atlas uses email; the mobile-app rule is H04's Boreal policy |

These are five development cases chosen for inspection, not a representative customer sample. The recovery query
is the same case as Part E, so its later call is not another independent question. Each loop retrieves ten vector
candidates, fuses that list with the keyword top ten, and separately reranks the vector candidates. We do not rerank
the fused list in this experiment. The first IDs printed are the candidates to inspect, not generated answers.


In [33]:
# atlas: five customer-worded questions; for each, the first ID from the vector, hybrid and reranked orders
atlas = [  # the recovery question from Lab 06 and four from Lab 04, each with the policy that answers it
    {"question": recovery, "known_support": ["H02"]},
    {"question": "How long does an Atlas reset link remain valid?", "known_support": ["H02"]},
    {"question": "Who can export all reports in Atlas?", "known_support": ["H03"]},
    {"question": "Can temporary Atlas accounts reset by email?", "known_support": ["H01"]},
    {"question": "Do I need the mobile app to reset my Atlas password?", "known_support": ["H01"]},
]
atlas_reranked = []
for q in atlas:
    found = lc_pipeline.retrieve(store, q["question"], k=10)
    ids = [document.id for document, _ in found]
    fused = rrf([ids, keyword_search(q["question"])])
    r = rerank(q["question"], found)
    atlas_reranked.append(r)
    check = "passed" if r["accepted"] else f"failed, left out {sorted(set(ids) - set(r['returned']))}"
    print(f"{q['known_support'][0]} | vector {ids[0]} | hybrid {fused[0]} | reranked {r['order'][0]}"
          f" | reply began {r['returned'][0]}, {check} | {q['question']}")

H02 | vector H01 | hybrid H01 | reranked H02 | reply began H02, passed | When does the password recovery email stop working for Atlas?


H02 | vector H02 | hybrid H02 | reranked H02 | reply began H02, passed | How long does an Atlas reset link remain valid?


H03 | vector H03 | hybrid H03 | reranked H03 | reply began H03, passed | Who can export all reports in Atlas?


H01 | vector H01 | hybrid H01 | reranked H01 | reply began H01, passed | Can temporary Atlas accounts reset by email?


H01 | vector H04 | hybrid H04 | reranked H04 | reply began H01, failed, left out ['H03'] | Do I need the mobile app to reset my Atlas password?


Each line: the right policy, the first ID of each order, and how the reranker's reply began and whether it passed the
ID check (a failed reply leaves the vector order in place).

The vector search and hybrid search put the right policy first for 3 of 5 and miss the same two. The reranker fixes the recovery question, the paraphrase both searches miss. For the mobile-app question its reply began with H01, the right policy, but left out H03, so the ID check kept the vector order (Challenge task 1 repairs such replies).

### Write the decision

A decision names the change, the questions it was measured on, what moved and what it costs:

| Change | Paired, 30 development questions | Right first, 5 Atlas questions | Cost per question | Decision |
|---|---|---|---|---|
| Hybrid search (BM25 + RRF) | hit@3: lost 1; hit@1: gained 1, lost 3 (p = 0.625) | 3 of 5 (vector 3 of 5) | a second index; 0.1 ms | Keep the vector search |
| Reranking the vector top 10 | hit@3: no change; hit@1: gained 1, lost 1 (p = 1.000); 2 replies of 30 failed the ID check | 4 of 5 | 961 tokens, 1,617 ms, 0.19 US dollars per 1,000 questions | Not yet; test on more customer questions |

As one line: *Keep the vector search. Hybrid search lost a question and gained none at hit@3; reranking gained as many as it lost, for about 960 tokens and 1.6 seconds more per question. Its one Atlas gain is a reason to collect more customer questions (Lab 06), not to ship it.*

### Checkpoint G

| You should see | Our run |
|---|---|
| Atlas questions | right first: vector 3 of 5, hybrid 3 of 5, reranked 4 of 5; the mobile-app reply began with H01 and failed, left out `['H03']` |

The decision uses source ranks and the application's fallback rule. It does not report answer correctness,
refusal quality or a calibrated probability of improvement. The five policy cases are too few and too closely
related for a deployment reliability claim. Next, collect fresh questions covering the distinctions that failed,
freeze their labels before testing, and evaluate the resulting answers as well as the retrieved IDs.


---

## Part H: Reuse the demonstrated search and comparison functions (self-study)

Part H moves this lab's functions into one module for the labs after this one: the BM25 index and search of Part B,
`rrf()`, `paired()`, `sign_test()` and `rerank()`, with its rules and `Ranking` class. Read it:


In [34]:
# the whole module, printed from shared/hybrid.py
import inspect
from shared import hybrid

print(inspect.getsource(hybrid))

"""Hybrid search, reranking and paired comparisons: the steps Lab 08 builds by hand, kept for the labs after it.

bm25_index() and bm25_search() rank records by the words they share with a question (BM25, through bm25s, set up
like the hand version: every word, no stop-word list, k1=1.2, b=0.75). rrf() fuses ranked ID lists by reciprocal
rank. rerank() asks RERANK_MODEL to order the candidates and keeps the search order unless the reply holds exactly the
candidate IDs. paired() and sign_test() compare a change with the baseline question by question.
"""
from __future__ import annotations

import time
from math import comb

import bm25s
from pydantic import BaseModel

from shared.settings import lc_chat_model

WORDS = r"(?u)\b\w+\b"      # every word, one letter or more, as terms() in Lab 08

RERANK_RULES = (
    "Rank the passages by how directly they answer the question."
    ' Reply in JSON shaped like {"ranking": ["ID", ...]}, with every passage ID once, best first.'
)


class Ranki

`hybrid.rerank()` builds its model inside, so it takes `settings`, and keeps the search order when the reply is not
exactly the candidates, as `rerank()` in Part F does. Use the module as a later lab will, and check it against what
you built:

The public BM25 constructor, tokenizer, indexing and search calls were used directly in Parts A–B; the chat
model's structured-output method was called in Part E. This module packages those demonstrated operations so later
labs can reuse them without repeating configuration. `hybrid.rerank()` also handles a missing parsed reply by
rejecting its empty ID list; it does not make a parsing failure into a successful ranking.

The first comparisons below are exact ID-list checks, including order. The sign-test comparison checks one known
input. The final line makes a new API call for the recovery query, so equal top IDs on that example do not prove
all future model replies will match the notebook's earlier call.


In [35]:
# the module's BM25 lists, fused lists, sign test and rerank, each compared with this lab's own
index = hybrid.bm25_index(corpus)
module_bm25 = [hybrid.bm25_search(index, q["question"]) for q in questions]
module_rrf = [hybrid.rrf([dense, keyword]) for dense, keyword in zip(dense_lists, module_bm25)]
print("same BM25 lists:", sum(a == b for a, b in zip(module_bm25, bm25_lists)), "of 30",
      "| same fused lists:", sum(a == b for a, b in zip(module_rrf, rrf_lists)), "of 30")
print("sign test, 1 gained and 3 lost:", hybrid.sign_test(1, 3), "| by hand:", sign_test(1, 3))
r = hybrid.rerank(recovery, candidates, settings)
print("recovery question reranked:", r["order"][:3], "| passed the ID check:", r["accepted"])

same BM25 lists: 30 of 30 | same fused lists: 30 of 30
sign test, 1 gained and 3 lost: 0.625 | by hand: 0.625


recovery question reranked: ['H02', 'H01', 'H04'] | passed the ID check: True


The deterministic parts return the same lists for these 30 inputs and the same p-value for the chosen split.
The additional model call puts H02 first and passes its ID check in this saved run. That verifies reuse on the
observed inputs; it is not a general equivalence test for model outputs, errors or other configurations.

### Checkpoint H

| You should see | Our run |
|---|---|
| Module results | same BM25 lists 30 of 30, same fused lists 30 of 30; p = 0.625 both ways; `['H02', 'H01', 'H04']`, passed |


---

## What you can do now

- Read the BM25 formula term by term, and say which words a keyword score came from.
- Fuse two ranked lists with reciprocal rank fusion, and explain why it sums reciprocal-rank contributions instead of raw search scores.
- Compare a change with the baseline in a paired table, name the questions it gained and lost, and run a sign test.
- Rerank candidates with a chat model and reject a reply that is not exactly the candidate IDs.
- Price a retrieval stage in tokens, dollars and seconds per question.
- Write a decision from the paired results and the cost, and say what evidence would change it.


## Going further: diverse evidence with maximal marginal relevance

Two of the three chunks an answer reads can say nearly the same thing. **Maximal marginal relevance** (MMR) picks each
next chunk by its similarity to the question minus its similarity to the chunks already picked.
`store.max_marginal_relevance_search(question, k=3, fetch_k=10)` takes the vector search's top 10 and picks three that
way:

`fetch_k=10` is the vector candidate depth and `k=3` the number selected. The store embeds the question again for
each call and returns a list of three `Document` objects, rather than `(Document, score)` pairs. `d.id` extracts
the IDs for our existing source-rank measure. The default tradeoff controls relevance versus similarity to already
selected documents; the call here leaves that setting unchanged. Diversity can remove repetition but can also
remove useful corroborating evidence, so a changed list needs answer-level evaluation.


In [36]:
# mmr_lists: three chunks per question picked by MMR from the top 10; compared with the vector search's top three
mmr_lists = [[d.id for d in store.max_marginal_relevance_search(q["question"], k=3, fetch_k=10)] for q in questions]
mmr_ranks = [first_hit(ids, q) for ids, q in zip(mmr_lists, questions)]
print(f"MMR hit@1 {sum(hits(mmr_ranks, 1))} of 30 | hit@3 {sum(hits(mmr_ranks, 3))} of 30"
      f" | top three differ from the vector top three: {sum(a != b[:3] for a, b in zip(mmr_lists, dense_lists))} of 30")

MMR hit@1 29 of 30 | hit@3 30 of 30 | top three differ from the vector top three: 30 of 30


MMR changes the top three of all 30 questions, and hit@1 (29) and hit@3 (30) stay the same: it keeps the most similar
chunk first. These totals do not reveal which other passages changed, or whether the answers improved. hit@k can detect a
change if it moves a known source across the cutoff; here it leaves those outcomes unchanged. Evaluate the actual
answer content and repetition too. Reranking the fused
list can be tested the same way, paired against the vector baseline.


## Challenge

Use only what this lab showed. Task 1 is the challenge; task 2 is optional. For each task, keep the
printed results and your answer to **Explain**.

1. **Repair a rejected ranking.** The ID check throws a whole reply away, even when an omitted candidate would not change the best result. A common repair keeps the reply's order for the IDs that are candidates, drops repeats and IDs that were
   not candidates, and adds the missing candidates at the end in the vector order.

   - Complete `repair()` under **Task 1**. The cell checks it on a made-up reply, then repairs every reply that failed
     in Parts F and G and prints the first ID before and after.

   **Check:** the made-up reply gives `['B', 'A', 'C', 'D']`. The worked solution moves the mobile-app question's first ID from H04 to H01 in its own saved run.
   Read the policy text to decide whether that helps; repairing the ID inventory does not establish relevance.

   **Explain:** was the changed first ID right? What could a repair hide that a rejection shows?

2. **Optional: check whether the candidate order sways the reranker.** Under **Task 2**, shuffle the recovery
   question's ten candidates five times (`random.Random(seed)`) and rerank each order.

   **Check:** how many of the five put H02 first. Our worked solution put H02 first in 5 of 5 orders, with H02 listed anywhere from second to tenth.

   **Explain:** what would you conclude if H02 came first only when it was listed first?


**Task 1.** Your `repair()`, then the check:

`returned` is the raw proposed order; `candidates` is the vector order that the fallback kept. Build a new list:
keep each candidate ID the first time it appears in `returned`, then append every still-missing candidate in original
order. Testing membership in a list is enough for ten candidates; no additional abstraction is needed.

For `["B", "A", "X", "B"]` and `["A", "B", "C", "D"]`, keep B and A, discard X and the repeated B, then append
C and D. The result contains the candidate inventory, but its preferred B can still be the wrong passage.
Keep a record of the rejected reply as well as the repaired order, so repair does not hide the model's failure.


In [37]:
def repair(returned, candidates):
    # TODO: the candidates in the reply's order (each once, no IDs that are not candidates),
    #       then the candidates the reply left out, in their own order
    return None

if repair(["B", "X", "B"], ["A", "B", "C", "D"]) is None:
    print("⚠️ Complete repair(), then run the cell again.")
else:
    print("made-up reply:", repair(["B", "A", "X", "B"], ["A", "B", "C", "D"]))
    failed = [(q, r) for q, r in zip(questions + atlas, reranked + atlas_reranked) if not r["accepted"]]
    for q, r in failed:
        fixed = repair(r["returned"], r["order"])
        print(f"{q['known_support']} | first before {r['order'][0]} | after {fixed[0]} | {q['question'].strip()}")

⚠️ Complete repair(), then run the cell again.


**Task 2 (optional).** Five shuffled orders of the recovery question's candidates:

`list(candidates)` makes a shallow copy of the hit list. Shuffling that copy changes the presentation order while
preserving the same Document objects and scores. `random.Random(seed)` gives a repeatable shuffle; it does not seed
the chat model or make its reply deterministic. `rerank()` still falls back to the supplied shuffled order on an
invalid reply, so inspect `accepted` and `returned` when interpreting a first ID. A fallback beginning with H02 is
not evidence that the model selected it.

The five calls test one fixed query under five presentations. An order effect would motivate a larger controlled
test, including repeated calls for each presentation; five successes cannot rule out sensitivity on other queries.
This optional task incurs five new rerank calls.


In [38]:
import random

firsts = []
for seed in range(5):
    shuffled = list(candidates)
    # TODO: shuffle `shuffled` with random.Random(seed), rerank it with rerank(recovery, shuffled),
    #       and append the first ID of the order to firsts

if not firsts:
    print("⚠️ Shuffle and rerank, then run the cell again.")
else:
    print("first ID per order:", firsts, "| H02 first:", firsts.count("H02"), "of", len(firsts))

⚠️ Shuffle and rerank, then run the cell again.


## Troubleshooting

| Symptom | Likely cause | Fix |
|---|---|---|
| `bm25s` scores in another order than the hand version (H02 last for the recovery question) | `bm25s` defaults: English stop words dropped, one-letter words ignored, `k1=1.5` | Pass `token_pattern=WORDS, stopwords=None` and `k1=1.2, b=0.75`, as `hybrid.bm25_index()` does |
| `ValueError: k of 10 is larger than the number of available scores, which is 5 (corpus size should be larger than top-k)...` | `retrieve(k=10)` on the five-policy index | Ask for at most as many results as the index holds |
| `OpenAIInvalidRequestError: Error code: 400 ... 'messages' must contain the word 'json' in some form, to use 'response_format' of type 'json_object'.` | `JSON_MODE=json_object` with a rule that never says JSON | Keep the word JSON in the rule, as `RULES` does |
| `OpenAIInvalidRequestError: Error code: 400 ... Unrecognized request argument supplied: reasoning_effort` on the first rerank call | `REASONING_EFFORT` set for a reasoning chat model while `RERANK_MODEL` is still `gpt-4o-mini` | Switch `CHAT_MODEL`, `JUDGE_MODEL` and `RERANK_MODEL` together |

## Resources

- [`bm25s` documentation](https://bm25s.github.io/): the BM25 package, its tokenizer options and scoring variants
- [Stanford IR book: Okapi BM25](https://nlp.stanford.edu/IR-book/html/htmledition/okapi-bm25-a-non-binary-model-1.html): term weighting, IDF, `k1` and `b`
- [Cormack, Clarke and Büttcher (2009)](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf): reciprocal rank
  fusion and `c = 60`
- [Elasticsearch: reciprocal rank fusion](https://www.elastic.co/docs/reference/elasticsearch/rest-apis/reciprocal-rank-fusion)
  and [Qdrant: hybrid queries](https://qdrant.tech/documentation/concepts/hybrid-queries/): hybrid search inside a
  search engine and a vector database
- [Sun and others (2023), RankGPT](https://arxiv.org/abs/2304.09542): listwise reranking with a chat model
- [Cohere: rerank overview](https://docs.cohere.com/docs/rerank-overview): a production cross-encoder reranker
- [LangChain v1 migration guide](https://docs.langchain.com/oss/python/migrate/langchain-v1): where the legacy
  retrievers moved

Next: [Lab 09: How to answer follow-up questions with retrieval](../09_Follow_Up_Questions/README.md). Back to
[the lab list](../README.md).

- [LangChain models: structured output](https://docs.langchain.com/oss/python/langchain/models#structured-output): parsed and raw replies from a model.
